## NodePort

In this notebook, we look at Services of type NodePort in Kubernetes.

- Make sure you have a Minikube Kubernetes cluster running, e.g. `minukube start`.
- Also make sure you have installed the `kubectl` tool on your computer.

## Deploy a Deployment

- The Deployment definition is in the YAML file `manifests/deploy-app.yaml`.

In [1]:
!kubectl apply -f manifests/deploy-app.yaml

deployment.apps/deploy-nginx created


## Let's look at the Deployment's YAML

**Note:**

- The number of `replicas` is set to 3.
- The `revisionHistoryLimit` is set to 3.
- The Pod template's container is listening on `containerPort` 80.
- The Pod template's `labels` are `app: nginx` and `env: prod`
  - The Deployment's `matchLabels` match these labels
    - Therefore, the Deployment will create 3 replicas from the Pod template.
  - The Service's `selector` matches these labels
    - Therefore, the Service will load balance between the three Pod replicas.

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: deploy-nginx         # the deployment is named deploy-nginx
spec:
  replicas: 3                 # the deployment defines three replicas
  revisionHistoryLimit: 3     # the deployment defines a maximum of three ReplicaSet revisions to store in history
  selector:
    matchLabels:
      app: nginx              # the deployment has two matchLabels that match the Pod template's labels below
      env: prod               # app = nginx and env = prod
  template:
    metadata:
      name: myapp-pod
      labels:
        app: nginx             # the Pod template has two labels defined (the Service's selector matches these two labels):
        env: prod              # app = nginx and env = prod
    spec:
      containers:
      - name: nginx
        image: nginx:alpine    # the container is based on the nginx:alpine image
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 250m
            memory: 256Mi    
        ports:
        - containerPort: 80    # the Pod template's container is listening on containerPort 80
```

In [2]:
#!type manifests\deploy-app.yaml # use this on Windows
!cat manifests/deploy-app.yaml # use this on Linux/Mac

apiVersion: apps/v1
kind: Deployment
metadata:
  name: deploy-nginx
  labels:
    app: nginx
    env: prod
spec:
  replicas: 3
  revisionHistoryLimit: 3
  selector:
    matchLabels:
      app: nginx
      env: prod
  template:
    metadata:
      labels:
        app: nginx
        env: prod
    spec:
      containers:
      - name: nginx
        image: nginx:alpine
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 250m
            memory: 256Mi
        ports:
        - containerPort: 80

## Deploy a Service

- The Service definition is in the YAML file `manifests/nodeport.yaml`.

In [3]:
!kubectl apply -f manifests/nodeport.yaml

service/svc-example created


## Let's look at the Service's YAML

**Note:**

- The service's name is `svc-example`.
- The service's is:
  - Listening for external (public) traffic on `nodePort` 32410.
  - Listening for internal (private) traffic on `port` 80.
  - Redirecting traffic to `targetPort` 80.
- The servie's `selector` defines two labels (that match the Pod template's labels for the Deployment):
  - `app: nginx` and `env: prod`
  - This means the Service will (round robin) load balance between the three Pods in the Deployment's ReplicaSet.

```bash
apiVersion: v1
kind: Service
metadata:
  name: svc-example    # the Service's name is svc-example
spec:
  type: NodePort       # the Service's type is NodePort
  selector:
    app: nginx         # the Service's selector defines two labels (the Service's selector matches the Pod template's two labels):
    env: prod          # app = nginx and env = prod
  ports:
  - nodePort: 32410    # the Service is listening for external (public) traffic on nodePort 32410
    protocol: TCP      # the Service's protocol is TCP
    port: 80           # the Service is listening for internal (private) traffic on port 80
    targetPort: 80     # the Service is redirecting traffic to targetPort 80
```

In [4]:
#!type manifests\nodeport.yaml # use this on Windows
!cat manifests/nodeport.yaml # use this on Linux/Mac

apiVersion: v1
kind: Service
metadata:
  name: svc-example
spec:
  type: NodePort
  selector:
    app: nginx
    env: prod
  ports:
  - nodePort: 32410
    protocol: TCP
    port: 80
    targetPort: 80

## List Pods

- We see that three Pods are running.

In [5]:
#!kubectl get po -o wide
!kubectl get pods -o wide

NAME                           READY   STATUS    RESTARTS   AGE   IP            NODE       NOMINATED NODE   READINESS GATES
deploy-nginx-6f689b949-fs557   1/1     Running   0          18s   10.244.0.27   minikube   <none>           <none>
deploy-nginx-6f689b949-nk7pr   1/1     Running   0          18s   10.244.0.26   minikube   <none>           <none>
deploy-nginx-6f689b949-tqjws   1/1     Running   0          18s   10.244.0.28   minikube   <none>           <none>


## List Services

- Notice the `TYPE` is `NodePort` for the `svc-example` Service (with `nodePort` = `32410`).
  - This means any of the nodes' IP addreses can be used to access the cluster externally (publicly) using port `32410`.

In [6]:
#!kubectl get svc -o wide
!kubectl get services -o wide

NAME          TYPE        CLUSTER-IP       EXTERNAL-IP   PORT(S)        AGE   SELECTOR
kubernetes    ClusterIP   10.96.0.1        <none>        443/TCP        80m   <none>
svc-example   NodePort    10.104.125.180   <none>        80:32410/TCP   18s   app=nginx,env=prod


- If the NodePort number was not explicitly defined (which is was in our YAML file), you can get the assigned NodePort number with the command below.

In [7]:
!kubectl get service svc-example --output='jsonpath="{.spec.ports[0].nodePort}"'

"32410"

## List Nodes

- Since we have a `NodePort` Service:
  - Any of the nodes' IP addresses (`INTERNAL-IP`) provide public access to the cluster using `nodePort` `32410`.
- In this case a single-node Minikube Kubernetes cluster is used, so there is only one node.

In [8]:
!kubectl get nodes -o wide

NAME       STATUS   ROLES           AGE   VERSION   INTERNAL-IP    EXTERNAL-IP   OS-IMAGE                         KERNEL-VERSION      CONTAINER-RUNTIME
minikube   Ready    control-plane   81m   v1.35.1   192.168.49.2   <none>        Debian GNU/Linux 12 (bookworm)   6.17.0-14-generic   docker://29.2.1


## Access Service of Type NodePort

- When using a cloud provider, you would use the cloud provider node's public IP address (`EXTERNAL-IP`) and NodePort.
  - Notice that the `EXTERNAL-IP` is `<none>` on our local computers.
- On a local Kubernetes cluster we would usually use any node's IP address (`INTERNAL-IP`) and NodePort number: `http://IP:NodePort`
  - Although, this won't work on Docker Desktop's Kubernetes cluster that runs inside a Virtual Machine.
- On a Minikube Kubernetes cluster, we use any node's IP address (`INTERNAL-IP`) and NodePort number: `http://IP:NodePort`

In [10]:
# Get internal IP of one node
INTERNAL_IP=!kubectl get nodes -o jsonpath='{.items[0].status.addresses[?(@.type=="InternalIP")].address}'
INTERNAL_IP = INTERNAL_IP[0]
print(f'INTERNAL_IP: {INTERNAL_IP}\n')

# Make HTTP request to INTERNAL_IP:NodePort
!curl http://$INTERNAL_IP:32410

INTERNAL_IP: 192.168.49.2

<!DOCTYPE html>
<html>
<head>
<title>Welcome to nginx!</title>
<style>
html { color-scheme: light dark; }
body { width: 35em; margin: 0 auto;
font-family: Tahoma, Verdana, Arial, sans-serif; }
</style>
</head>
<body>
<h1>Welcome to nginx!</h1>
<p>If you see this page, nginx is successfully installed and working.
Further configuration is required for the web server, reverse proxy, 
API gateway, load balancer, content cache, or other features.</p>

<p>For online documentation and support please refer to
<a href="https://nginx.org/">nginx.org</a>.<br/>
To engage with the community please visit
<a href="https://community.nginx.org/">community.nginx.org</a>.<br/>
For enterprise grade support, professional services, additional 
security features and capabilities please refer to
<a href="https://f5.com/nginx">f5.com/nginx</a>.</p>

<p><em>Thank you for using nginx.</em></p>
</body>
</html>


## Delete the Service and Deployment

In [11]:
!kubectl delete -f manifests/nodeport.yaml
!kubectl delete -f manifests/deploy-app.yaml

service "svc-example" deleted from default namespace
deployment.apps "deploy-nginx" deleted from default namespace


## List Services, Deployments and Pods

- We see that the Service and Deployment (with its three Pods) have been deleted.
  - The service with **NAME** `kubernetes` and **TYPE** `ClusterIP` is for routing requests to the Kubernetes API Server, and should never be deleted.

In [12]:
#!kubectl get svc -o wide
#!kubectl get deploy -o wide
#!kubectl get po -o wide
!kubectl get services -o wide
!kubectl get deployments -o wide
!kubectl get pods -o wide

NAME         TYPE        CLUSTER-IP   EXTERNAL-IP   PORT(S)   AGE   SELECTOR
kubernetes   ClusterIP   10.96.0.1    <none>        443/TCP   82m   <none>
No resources found in default namespace.
No resources found in default namespace.
